# eBay Laptops & Netbooks - Modeling

In this project, we aim to build a predictive model for **estimating laptop prices** using a dataset which contains cleaned information from eBay's Laptops & Netbooks category, originally obtained via web scraping, which includes product attributes such as brand, specifications, and other listing details.

The model utilizes **CatBoost** (*Categorical Boosting*), a state-of-the-art gradient boosting library [developed by Yandex][Yandex CatBoost], renowned for its efficiency in handling categorical features and its strong performance in regression, classification & ranking.

<div align="center">
<img src="../assets/logos/catboost_logo.png" height="200" width="200"/>
</div>

[Yandex Catboost]: https://yandex.com/dev/catboost/

## Links & Information

**Project Repository** - GitHub: [Laptop Price Prediction with CatBoost][Project Code]

[Project Code]: https://github.com/jxareas/laptop-price-catboost


### Loading Libraries



In [1]:
# Importing libraries and setting constants

# Libraries
import polars as pl

# Constants
RANDOM_SEED = 287
DATA_SOURCE_PATH = '../data/ebay_laptops_and_netbooks_cleansed.csv'



## Data Preparation

### Loading the dataset

In [2]:
df = pl.read_csv(DATA_SOURCE_PATH)
df.head(n=10).to_pandas()

,brand,currency,min_price,rating,five_star_scale_rating,ratings_count,condition_label,condition_description,seller_note,processor,...,processor_speed_unit,laptop_type,release_year,display_width,display_height,model,os,features,country_of_manufacturer,storage_type
0,unknown,$,59.42,None,NaN,NaN,no_label,No description,None,None,...,unknown,unknown,NaN,NaN,NaN,None,unknown,None,unknown,unknown
1,unknown,$,65.08,None,NaN,NaN,new,"A brand-new, unused, unopened, undamaged item ...",None,None,...,unknown,unknown,NaN,NaN,NaN,None,unknown,None,unknown,unknown
2,lenovo,$,55.99,5_out_of_5_stars,5.0,3.0,no_label,No description,None,mediatek,...,GHz,laptop,NaN,1366.0,768.0,lenovo_100e,chrome,None,unknown,emmc
3,unknown,$,103.73,None,NaN,NaN,no_label,No description,None,None,...,unknown,unknown,NaN,NaN,NaN,None,unknown,None,unknown,unknown
4,lenovo,$,60.97,None,NaN,NaN,new,"A brand-new, unused, unopened, undamaged item ...",None,does_not_apply,...,unknown,unknown,NaN,NaN,NaN,None,unknown,None,unknown,unknown
5,acer,$,273.03,None,NaN,NaN,open_box,"An item in excellent, new condition with no we...",“This laptop has been FACTORY REFURBISHED by A...,amd_ryzen_3,...,GHz,laptop,NaN,1920.0,1080.0,i_pus,windows,"thin&light,backlitkeyboard,bluetooth,built-inm...",unknown,ssd
6,dell,$,467.52,None,NaN,NaN,used,An item that has been used previously.\n Th...,None,intel_core_i5_4th_generation,...,GHz,laptop,NaN,1280.0,720.0,e5440,windows,"10/100lancard,wi-fi,sdcardreader",unknown,unknown
7,unknown,$,52.73,None,NaN,NaN,new,"A brand-new, unused, unopened, undamaged item ...",None,None,...,unknown,unknown,NaN,NaN,NaN,None,unknown,None,unknown,unknown
8,apple,$,549.00,None,NaN,NaN,used,An item that has been used previously.\n Th...,None,intel_core_i7_8th_gen,...,unknown,laptop,2019.0,2560.0,1600.0,mv982lla,mac,None,unknown,ssd
9,hp,$,239.98,None,NaN,NaN,used,An item that has been used previously.\n Th...,"“Excellent condition, very minimal signs of we...",amd_ryzen_7,...,GHz,laptop,NaN,1920.0,1080.0,elitebook_745_g6,windows,"backlitkeyboard,bluetooth,built-inmicrophone,b...",unknown,ssd


### Dropping duplicates

In [3]:
df = df.unique()
df.head(n=10).to_pandas()

,brand,currency,min_price,rating,five_star_scale_rating,ratings_count,condition_label,condition_description,seller_note,processor,...,processor_speed_unit,laptop_type,release_year,display_width,display_height,model,os,features,country_of_manufacturer,storage_type
0,fujitsu_siemens,$,434.98,None,NaN,NaN,used,An item that has been used previously.\n Th...,None,intel_core_i5_4th_generation,...,GHz,laptop,NaN,1280.0,720.0,a574,windows,"10/100lancard,opticaldrive,wi-fi,bluetooth",unknown,ssd
1,unknown,$,131.21,None,NaN,NaN,new,"A brand-new, unused, unopened, undamaged item ...",None,None,...,unknown,unknown,NaN,NaN,NaN,None,unknown,None,unknown,unknown
2,hp,$,200.00,None,NaN,NaN,used,An item that has been used previously.\n Th...,“5 Chromebooks. AC adapters included. Google C...,amd_a4_dual_core,...,GHz,laptop,2019.0,NaN,NaN,hp_14a_g5,chrome,"bluetooth,built-inmicrophone,built-inwebcam,wi-fi",unknown,ssd
3,unknown,$,485.58,None,NaN,NaN,new,"A brand-new, unused, unopened, undamaged item ...",None,None,...,unknown,unknown,NaN,NaN,NaN,None,unknown,None,unknown,unknown
4,unknown,$,104.98,None,NaN,NaN,new,"A brand-new, unused, unopened, undamaged item ...",None,None,...,unknown,unknown,NaN,NaN,NaN,None,unknown,None,unknown,unknown
5,dell,$,1014.49,None,NaN,NaN,used,An item that has been used previously.\n Th...,None,intel_core_m5,...,unknown,laptop,NaN,1280.0,720.0,12_rugged_tablets,windows,webcamintegrated,unknown,unknown
6,gpd,$,348.99,None,NaN,NaN,used,An item that has been used previously.\n Th...,“Please be sure to read the description before...,intel_core_m3_7th_gen,...,GHz,laptop,NaN,1920.0,1200.0,gdp_pocket_2,windows,touchscreen,china,emmc
7,unknown,$,114.02,None,NaN,NaN,new,"A brand-new, unused, unopened, undamaged item ...",None,None,...,unknown,unknown,NaN,NaN,NaN,None,unknown,None,unknown,unknown
8,dell,$,44.97,None,NaN,NaN,used,An item that has been used previously.\n Th...,None,intel_core_2_duo,...,GHz,laptop,NaN,1920.0,1200.0,dell_latitude_e6500,other,"opticaldrive,wi-fi",unknown,unknown
9,unknown,$,1030.10,None,NaN,NaN,seller_refurbished,The item has been restored to working order by...,None,None,...,unknown,unknown,NaN,NaN,NaN,None,unknown,None,unknown,unknown


## Exploratory Data Analysis

In [4]:
# TODO : Exploratory Data Analysis

## Feature Engineering

In [5]:
# TODO : Feature Engineering

## Machine Learning

In [6]:
# TODO : Machine Learning

## Hyperparameter Tuning

In [7]:
# TODO : Hyperparameter Tuning

## Explainable AI - SHAP

In [8]:
# TODO: Explainable AI - SHAP